# Atividade em Python — Misturas
**Termodinâmica Avançada — 2026.2 · Aula 09**

Cinco exercícios, todos com o mesmo par benzeno + tolueno a 25 °C que a aula
usou. **Cada um termina num `assert` com o número que apareceu na tela**: se a
célula imprimir `ok`, está certo — não é preciso esperar correção.

Entregue este caderno executado, ao final da aula.

## Preparo
Rode esta célula primeiro. Ela traz as constantes e a equação de Antoine,
que calcula a pressão de vapor do líquido puro — a mesma ferramenta da
Aula 07.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

R  = 8.314          # J/(mol K)
T  = 298.15         # K   (25 °C, a temperatura da aula inteira)
RT = R * T          # J/mol

# Antoine:  log10 p(mmHg) = A - B/(C + t/°C)
ANTOINE = {"benzeno": (6.90565, 1211.033, 220.790),
           "tolueno": (6.95464, 1344.800, 219.482)}
MMHG_KPA = 101.325 / 760.0

def psat(nome, t=25.0):
    """Pressão de vapor do líquido PURO, em kPa, na temperatura t (°C)."""
    A, B, C = ANTOINE[nome]
    return 10.0 ** (A - B / (C + t)) * MMHG_KPA

pA = psat("benzeno")    # o mais volátil
pB = psat("tolueno")

print(f"RT = {RT/1000:.3f} kJ/mol")
print(f"p*(benzeno, 25 °C) = {pA:.2f} kPa")
print(f"p*(tolueno, 25 °C) = {pB:.2f} kPa")

## 1. O termo $RT\ln x$

No vídeo da poça, o custo de o vapor estar a uma pressão parcial mais baixa
que a de saturação era

$$\Delta\mu = RT\ln(\mathrm{UR}),$$

com UR a umidade relativa. Escreva `delta_mu(ur)` e reproduza a tabela da
aula.

**Esperado:** $0$, $-0{,}55$, $-1{,}72$ e $-3{,}99$ kJ/mol.

In [ ]:
def delta_mu(ur):
    """Δμ do vapor de água no ar, em J/mol, para umidade relativa `ur`."""
    # SEU CÓDIGO AQUI (uma linha)
    raise NotImplementedError

for ur in (1.00, 0.80, 0.50, 0.20):
    print(f"UR = {ur*100:3.0f} %   Δμ = {delta_mu(ur)/1000:+.2f} kJ/mol")

assert abs(delta_mu(1.00)) < 1e-9
assert abs(delta_mu(0.50)/1000 - (-1.72)) < 0.005
assert abs(delta_mu(0.20)/1000 - (-3.99)) < 0.005
print("ok")

## 2. A energia livre de mistura

Para uma binária ideal,

$$\Delta g_{\mathrm{mist}} = RT\,(x\ln x + (1-x)\ln(1-x)),
\qquad
\Delta s_{\mathrm{mist}} = -R\,(x\ln x + (1-x)\ln(1-x)).$$

Escreva as duas funções, trace as duas curvas e confira os três números do
vídeo no ponto meio a meio.

**Esperado:** $\Delta g = -1{,}718$ kJ/mol, $\Delta s = R\ln 2 = 5{,}763$
J/(mol K) e $\Delta h = \Delta g + T\Delta s = 0$ — zero **exato**, por
cancelamento.

In [ ]:
def dg_mix(x):
    """Δg de mistura de uma binária ideal, em J/mol."""
    # SEU CÓDIGO AQUI
    raise NotImplementedError

def ds_mix(x):
    """Δs de mistura de uma binária ideal, em J/(mol K)."""
    # SEU CÓDIGO AQUI
    raise NotImplementedError

xx = np.linspace(1e-9, 1-1e-9, 600)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(xx, dg_mix(xx)/1000, color="#A16207")
ax[0].set_ylabel(r"$\Delta g_{mist}$  (kJ/mol)")
ax[1].plot(xx, ds_mix(xx), color="#7E22CE")
ax[1].set_ylabel(r"$\Delta s_{mist}$  (J/(mol K))")
for a in ax:
    a.set_xlabel(r"$x_A$")
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Δg(0,5) = {dg_mix(0.5)/1000:.3f} kJ/mol")
print(f"Δs(0,5) = {ds_mix(0.5):.3f} J/(mol K)   e  R ln2 = {R*np.log(2):.3f}")
print(f"Δh = Δg + TΔs = {dg_mix(0.5) + T*ds_mix(0.5):.3e} J/mol")

assert abs(dg_mix(0.5)/1000 - (-1.718)) < 0.002
assert abs(ds_mix(0.5) - R*np.log(2)) < 1e-9
assert abs(dg_mix(0.5) + T*ds_mix(0.5)) < 1e-9
print("ok")

## 3. Antoine, e a temperatura de ebulição

A pressão de vapor do puro é a única coisa que Raoult precisa saber sobre a
substância. A temperatura de ebulição normal é onde ela chega a 1 atm
($101{,}325$ kPa): ache-a para os dois, resolvendo
$p^{*}(t) = 101{,}325$ com `brentq`.

**Esperado:** benzeno $80{,}1$ °C e tolueno $110{,}6$ °C — os dois valores
tabelados. É a aferição das constantes de Antoine que você vai usar no
exercício seguinte.

In [ ]:
def t_ebulicao(nome, p_alvo=101.325):
    """Temperatura (°C) em que a pressão de vapor do puro vale `p_alvo` kPa."""
    # SEU CÓDIGO AQUI --- use brentq entre 10 e 200 °C
    raise NotImplementedError

for nome in ("benzeno", "tolueno"):
    print(f"{nome:8s}  p*(25 °C) = {psat(nome):5.2f} kPa"
          f"   T_eb = {t_ebulicao(nome):.1f} °C")

assert abs(t_ebulicao("benzeno") - 80.1) < 0.1
assert abs(t_ebulicao("tolueno") - 110.6) < 0.1
print("ok")

## 4. O diagrama $p$–$x$, e a lente

Com Raoult em cada componente e Dalton somando,

$$p = p_B^{*} + (p_A^{*}-p_B^{*})\,x_A,
\qquad
y_A = \frac{x_A\,p_A^{*}}{p}.$$

Trace as duas curvas **no mesmo eixo** — a de bolha contra $x$, a de orvalho
contra $y$ — e marque a horizontal que liga as duas em $x_A = 0{,}500$.

**Esperado:** $p = 8{,}24$ kPa e $y_A = 0{,}770$. O vapor tem 77 % de benzeno
onde o líquido tem 50 %.

In [ ]:
def p_total(x):
    """Pressão total sobre um líquido de fração molar x de benzeno (kPa)."""
    # SEU CÓDIGO AQUI
    raise NotImplementedError

def y_vapor(x):
    """Fração molar de benzeno NO VAPOR em equilíbrio com esse líquido."""
    # SEU CÓDIGO AQUI
    raise NotImplementedError

x  = np.linspace(0.0, 1.0, 400)
x0 = 0.500
p0, y0 = p_total(x0), y_vapor(x0)

fig, ax = plt.subplots(figsize=(6.6, 4.3))
ax.fill(np.concatenate([x, y_vapor(x)[::-1]]),
        np.concatenate([p_total(x), p_total(x)[::-1]]),
        color="#A16207", alpha=0.15)
ax.plot(x, p_total(x), color="#1D4ED8", lw=2.4, label="bolha (líquido)")
ax.plot(y_vapor(x), p_total(x), color="#15803D", lw=2.4,
        label="orvalho (vapor)")
ax.hlines(p0, x0, y0, color="#111827", ls="--", lw=1.6)
ax.plot([x0, y0], [p0, p0], "o", color="#111827")
ax.set_xlabel("fração molar de benzeno")
ax.set_ylabel("p  (kPa)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"x = {x0:.3f}   ->   p = {p0:.2f} kPa,   y = {y0:.3f}")

assert abs(p0 - 8.24) < 0.01
assert abs(y0 - 0.770) < 0.001
print("ok")

## 5. Quantos estágios?

É daí que vem a destilação: o vapor é sempre mais rico no componente mais
volátil. Condense o vapor, deixe evaporar de novo, e repita — cada repetição
é um **estágio**.

Partindo de um líquido com $x_A = 0{,}500$, quantos estágios são precisos
para passar de 99 % de benzeno?

**Esperado:** 4 estágios — $0{,}770 \to 0{,}918 \to 0{,}974 \to 0{,}992$.

In [ ]:
x = 0.500
estagios = []
# SEU CÓDIGO AQUI: repita "o vapor vira o próximo líquido" até passar de 0,99,
# guardando cada composição na lista `estagios`.
raise NotImplementedError

for i, xi in enumerate(estagios, 1):
    print(f"estágio {i}:  y = {xi:.4f}")
print(f"\nbastam {len(estagios)} estágios para passar de 99 % de benzeno")

assert len(estagios) == 4
assert abs(estagios[0] - 0.770) < 0.001
assert estagios[-1] > 0.99
print("ok")

## Para entregar

Este caderno executado, com as cinco células imprimindo `ok`, e as duas
respostas abaixo escritas numa célula de texto.

1. No exercício 2, $\Delta h_{\mathrm{mist}} = 0$ saiu **exatamente** zero,
   e não aproximadamente. De onde vem esse zero: do número ou da álgebra?
   Em que situação física ele deixaria de valer?

2. No exercício 5, a composição se aproxima de 1 mas nunca chega. Olhando o
   diagrama do exercício 4, o que acontece com a distância entre a curva de
   bolha e a de orvalho quando $x_A \to 1$ — e o que isso diz sobre o custo
   de purificar os últimos décimos de por cento?